# Phase 2 Rolling Feature Check

In [ ]:
from pathlib import Path
import gc

import lightgbm as lgb
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / "data" / "top20_p8"
TARGET_COL = "responder_6"
WEIGHT_COL = "weight"
DATE_COL = "date_id"
TIME_COL = "time_id"
SYMBOL_COL = "symbol_id"
CATEGORICAL_COLS = ["symbol_id", "feature_11"]
SEED = 42

ROLL_WINDOWS = [5, 20]
ROLL_BASE_FEATURES = [
    "feature_08", "feature_36", "feature_61", "feature_04", "feature_20",
    "feature_06", "feature_24", "feature_29", "feature_58", "feature_23",
]

TRAIN_DATE_COUNT = 30
VALID_DATE_COUNT = 10

In [ ]:
def weighted_r2(y_true, y_pred, weight):
    y_true = np.asarray(y_true, dtype=np.float64)
    y_pred = np.asarray(y_pred, dtype=np.float64)
    weight = np.asarray(weight, dtype=np.float64)
    numerator = np.sum(weight * np.square(y_true - y_pred))
    denominator = np.sum(weight * np.square(y_true))
    return np.nan if denominator == 0 else 1.0 - numerator / denominator


def load_phase2_sample():
    train = pd.read_parquet(DATA_DIR / "train.parquet")
    valid = pd.read_parquet(DATA_DIR / "valid.parquet")
    df = pd.concat([train, valid], ignore_index=True)
    df = df.sort_values([DATE_COL, TIME_COL, SYMBOL_COL]).reset_index(drop=True)

    dates = sorted(df[DATE_COL].unique())
    selected_dates = dates[-(TRAIN_DATE_COUNT + VALID_DATE_COUNT):]
    df = df[df[DATE_COL].isin(selected_dates)].copy().reset_index(drop=True)

    train_dates = selected_dates[:TRAIN_DATE_COUNT]
    valid_dates = selected_dates[TRAIN_DATE_COUNT:]
    return df, train_dates, valid_dates


def feature_cols(df):
    return [col for col in df.columns if col not in [DATE_COL, WEIGHT_COL, TARGET_COL]]


def cast_categoricals(df):
    for col in CATEGORICAL_COLS:
        if col in df.columns:
            df[col] = df[col].astype("category")
    return df

In [ ]:
df, train_dates, valid_dates = load_phase2_sample()
base_features = feature_cols(df)
roll_base_features = [col for col in ROLL_BASE_FEATURES if col in df.columns]

print({
    "shape": df.shape,
    "train_dates": (int(min(train_dates)), int(max(train_dates))),
    "valid_dates": (int(min(valid_dates)), int(max(valid_dates))),
    "base_features": len(base_features),
    "rolling_base_features": roll_base_features,
})

df.head()

In [ ]:
def add_symbol_rolling_features(df, features, windows):
    out = df.sort_values([SYMBOL_COL, DATE_COL, TIME_COL]).copy()
    created = []

    grouped = out.groupby(SYMBOL_COL, observed=True, sort=False)
    for feature in features:
        shifted = grouped[feature].shift(1)
        for window in windows:
            mean_col = f"{feature}_sym_roll{window}_mean"
            std_col = f"{feature}_sym_roll{window}_std"
            out[mean_col] = shifted.groupby(out[SYMBOL_COL], observed=True).rolling(window, min_periods=2).mean().reset_index(level=0, drop=True)
            out[std_col] = shifted.groupby(out[SYMBOL_COL], observed=True).rolling(window, min_periods=2).std().reset_index(level=0, drop=True)
            created.extend([mean_col, std_col])

    out[created] = out[created].fillna(0).astype("float32")
    out = out.sort_values([DATE_COL, TIME_COL, SYMBOL_COL]).reset_index(drop=True)
    return out, created


roll_df, rolling_features = add_symbol_rolling_features(df, roll_base_features, ROLL_WINDOWS)
print({"rolling_feature_count": len(rolling_features), "shape": roll_df.shape})
roll_df[rolling_features].head()

In [ ]:
def train_lgbm(train_df, valid_df, features):
    train_df = cast_categoricals(train_df.copy())
    valid_df = cast_categoricals(valid_df.copy())
    categorical_features = [col for col in CATEGORICAL_COLS if col in features]

    train_set = lgb.Dataset(
        train_df[features],
        label=train_df[TARGET_COL],
        weight=train_df[WEIGHT_COL],
        feature_name=features,
        categorical_feature=categorical_features,
        free_raw_data=False,
    )
    valid_set = lgb.Dataset(
        valid_df[features],
        label=valid_df[TARGET_COL],
        weight=valid_df[WEIGHT_COL],
        feature_name=features,
        categorical_feature=categorical_features,
        free_raw_data=False,
    )

    params = {
        "objective": "regression",
        "metric": "rmse",
        "boosting_type": "gbdt",
        "device_type": "cpu",
        "num_leaves": 64,
        "learning_rate": 0.03,
        "min_data_in_leaf": 500,
        "feature_fraction": 0.85,
        "bagging_fraction": 0.85,
        "bagging_freq": 1,
        "lambda_l2": 3.0,
        "seed": SEED,
        "feature_pre_filter": False,
        "verbosity": -1,
    }

    model = lgb.train(
        params,
        train_set,
        valid_sets=[train_set, valid_set],
        valid_names=["train", "valid"],
        num_boost_round=600,
        callbacks=[lgb.early_stopping(50), lgb.log_evaluation(50)],
    )
    pred = model.predict(valid_df[features], num_iteration=model.best_iteration)
    score = weighted_r2(valid_df[TARGET_COL], pred, valid_df[WEIGHT_COL])
    return model, score, pred

In [ ]:
train_mask = roll_df[DATE_COL].isin(train_dates)
valid_mask = roll_df[DATE_COL].isin(valid_dates)

train_base = roll_df.loc[train_mask].copy()
valid_base = roll_df.loc[valid_mask].copy()

base_model, base_score, base_pred = train_lgbm(train_base, valid_base, base_features)
rolling_model, rolling_score, rolling_pred = train_lgbm(train_base, valid_base, base_features + rolling_features)

comparison = pd.DataFrame([
    {"model": "base_top20", "valid_weighted_r2": base_score, "features": len(base_features), "best_iteration": base_model.best_iteration},
    {"model": "top20_plus_rolling", "valid_weighted_r2": rolling_score, "features": len(base_features + rolling_features), "best_iteration": rolling_model.best_iteration},
])
comparison

In [ ]:
plt.figure(figsize=(8, 4))
sns.barplot(data=comparison, x="model", y="valid_weighted_r2", color="steelblue")
plt.axhline(0, color="black", linewidth=1)
plt.title("Validation Weighted R2: Base vs Rolling Features")
plt.xticks(rotation=15)
plt.tight_layout()

In [ ]:
importance = (
    pd.DataFrame({
        "feature": rolling_model.feature_name(),
        "importance_gain": rolling_model.feature_importance(importance_type="gain"),
        "importance_split": rolling_model.feature_importance(importance_type="split"),
    })
    .sort_values(["importance_gain", "importance_split"], ascending=False)
    .reset_index(drop=True)
)

plt.figure(figsize=(10, 7))
sns.barplot(data=importance.head(25), y="feature", x="importance_gain", color="steelblue")
plt.title("Rolling Model Feature Importance")
plt.tight_layout()

importance.head(25)

In [ ]:
valid_plot = valid_base[[DATE_COL, TIME_COL, SYMBOL_COL, TARGET_COL, WEIGHT_COL]].copy()
valid_plot["base_pred"] = base_pred
valid_plot["rolling_pred"] = rolling_pred

rows = []
for date_id, group in valid_plot.groupby(DATE_COL):
    rows.append({
        "date_id": date_id,
        "base_r2": weighted_r2(group[TARGET_COL], group["base_pred"], group[WEIGHT_COL]),
        "rolling_r2": weighted_r2(group[TARGET_COL], group["rolling_pred"], group[WEIGHT_COL]),
    })
by_date = pd.DataFrame(rows)

plt.figure(figsize=(12, 4))
sns.lineplot(data=by_date, x="date_id", y="base_r2", marker="o", label="base")
sns.lineplot(data=by_date, x="date_id", y="rolling_r2", marker="o", label="rolling")
plt.axhline(0, color="black", linewidth=1)
plt.title("Validation Weighted R2 by Date")
plt.tight_layout()

by_date